# Лабораторная 2: LSTM для задачи many-to-many 

## Цель
Построить `LSTM`-модель, которая выдает бинарную метку на каждом шаге времени.

Формат `many-to-many` означает: на каждом временном шаге формируется собственное предсказание.

## Контракт данных
Используется учебная синтетическая постановка:
- вход: последовательность из `-1` и `+1`;
- метка на шаге `t`: знак кумулятивной суммы `x_1 + ... + x_t`.

Таким образом, правильная метка каждого шага зависит от предыдущей истории.


## Таблица форм тензоров

| Тензор | Форма | Смысл | Где используется |
|---|---|---|---|
| `X` | `(N, T, 1)` | Входные последовательности | Генерация данных |
| `cumsum` | `(N, T)` | Промежуточная кумулятивная сумма | Генерация меток |
| `y` | `(N, T, 1)` | Метки для каждого шага | Обучение и оценка |
| `probs` | `(N_test, T, 1)` | Вероятности класса по шагам | `model.predict` |
| `preds` | `(N_test, T, 1)` | Бинарные предсказания по шагам | Оценка |
| `token_accuracy` | скаляр | Доля верных токенов | Метрика модели |
| `sequence_accuracy` | скаляр | Доля полностью верных последовательностей | Доп. метрика |


## Контракт модели
- В `model.fit` передается `X_train` и `y_train` одинаковой длины по времени.
- Рекуррентный слой должен возвращать всю последовательность состояний (`return_sequences=True`).
- Выход `model.predict` имеет ту же временную длину, что и вход.
- Потеря: бинарная кросс-энтропия (`binary_crossentropy`) по всем временным шагам.


## Мини-теория
В канонической нотации `LSTM`:

$$
u_t=[h_{t-1};x_t], \quad
f_t = \sigma(W_f u_t + b_f), \quad
i_t = \sigma(W_i u_t + b_i), \quad
\tilde{c}_t = \tanh(W_c u_t + b_c), \quad
o_t = \sigma(W_o u_t + b_o)
$$

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t, \quad
h_t = o_t \odot \tanh(c_t), \quad
s_t = W_{hy}h_t + b_y, \quad
\hat{y}_t = \sigma(s_t)
$$

Формат `many-to-many` реализуется через выдачу предсказаний на каждом шаге времени.


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    tf.random.set_seed(seed)


set_seed(42)
print('Версия TensorFlow:', tf.__version__)

## Генерация данных
**Что сделать:** построить покадровые метки на основе кумулятивной суммы.

**Почему:** каждая метка должна зависеть от предыдущих шагов.

**Ожидаемые формы:** `X -> (N,T,1)`, `y -> (N,T,1)`.

**Частая ошибка:** формировать `y` как `(N,T)` без последней оси.


In [ ]:
def make_dataset(n_samples: int = 6500, seq_len: int = 16):
    x = np.random.choice([-1.0, 1.0], size=(n_samples, seq_len, 1)).astype(np.float32)
    # TODO 1: посчитайте кумулятивную сумму по временной оси
    cumsum = ...
    # TODO 2: сформируйте бинарные метки формы (N,T,1)
    y = ...
    return x, y, cumsum


X, y, cumsum = make_dataset()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print('Форма X_train:', X_train.shape)
print('Форма y_train:', y_train.shape)
print('Форма X_test :', X_test.shape)
print('Форма y_test :', y_test.shape)

In [ ]:
# Мини-проверка данных
assert X.ndim == 3 and X.shape[-1] == 1, 'Ожидается X формы (N,T,1)'
assert y.ndim == 3 and y.shape[-1] == 1, 'Ожидается y формы (N,T,1)'
assert cumsum.shape == X[:, :, 0].shape, 'Форма cumsum должна быть (N,T)'
print('Мини-проверка данных: OK')

## Модель
**Что сделать:** собрать `Sequential`-модель с `LSTM(return_sequences=True)`.

**Почему:** нужен выход на каждом временном шаге.

**Ожидаемая форма выхода:** `(batch, T, 1)`.


In [ ]:
def build_model(input_shape):
    model = tf.keras.Sequential(name='lstm_many_to_many')
    model.add(tf.keras.layers.Input(shape=input_shape))
    # TODO 3: добавьте слой LSTM с return_sequences=True
    model.add(...)
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    # TODO 4: скомпилируйте модель (adam, binary_crossentropy, accuracy)
    model.compile(...)
    return model


model = build_model(X_train.shape[1:])
model.summary()

In [ ]:
# Мини-проверка модели
assert model.output_shape[1:] == y_train.shape[1:], 'Форма выхода должна совпадать с формой y'
print('Мини-проверка модели: OK')

## Трассировка одного примера через модель
Проверка форм тензоров:
1. входной пример;
2. выход `LSTM`-слоя на всех шагах;
3. итоговый выходной слой.


In [ ]:
sample_x = X_train[:1]
_ = model(sample_x)

lstm_features_model = tf.keras.Model(inputs=model.inputs[0], outputs=model.layers[-2].output)
lstm_features = lstm_features_model.predict(sample_x, verbose=0)

sample_prob = model.predict(sample_x, verbose=0)

print('Вход sample_x      :', sample_x.shape)
print('После LSTM         :', lstm_features.shape)
print('После выходного слоя:', sample_prob.shape)
print('Первые 3 вероятности:', sample_prob[0, :3, 0])

## Как идет обучение внутри эпохи
Для каждого мини-батча:
1. прямой проход (`forward pass`) по всем шагам;
2. вычисление покадровой функции потерь;
3. обратное распространение через время (`BPTT`);
4. обновление параметров.

Для `LSTM` поток градиента стабилизируется за счет памяти `c_t` и ворот `f_t`, `i_t`, `o_t`, поэтому обычно легче обучать зависимости средней и большой длины.


## Обучение
Обучение выполняется с `validation_split=0.2` на обучающей части. Контрольная выборка `test` не участвует в подборе параметров.


In [ ]:
def train_model(model, X_train, y_train, epochs: int = 12):
    # TODO 5: обучите модель с validation_split=0.2 и batch_size=64
    history = model.fit(...)
    return history


history = train_model(model, X_train, y_train)

In [ ]:
# Мини-проверка обучения
assert 'val_loss' in history.history and 'val_accuracy' in history.history
print('Последняя val_accuracy:', float(history.history['val_accuracy'][-1]))
print('Мини-проверка обучения: OK')

## Оценка и диагностика
Считаются две метрики:
- `token_accuracy`: доля корректных шагов времени;
- `sequence_accuracy`: доля последовательностей, где корректны все шаги.

Вторая метрика более строгая и чувствительна к единичным ошибкам.


In [ ]:
def evaluate_model(model, X_test, y_test):
    loss, token_acc = model.evaluate(X_test, y_test, verbose=0)
    probs = model.predict(X_test, verbose=0)
    # TODO 6: преобразуйте вероятности в бинарные предсказания порогом 0.5
    preds = ...
    # TODO 7: вычислите sequence_accuracy как долю полностью верных последовательностей
    seq_acc = ...
    return {
        'loss': float(loss),
        'token_accuracy': float(token_acc),
        'sequence_accuracy': float(seq_acc),
        'probs': probs,
        'preds': preds,
    }


metrics = evaluate_model(model, X_test, y_test)
print({k: v for k, v in metrics.items() if k in ('loss', 'token_accuracy', 'sequence_accuracy')})

In [ ]:
# Мини-проверка метрик
assert 0.0 <= metrics['token_accuracy'] <= 1.0
assert 0.0 <= metrics['sequence_accuracy'] <= 1.0
if metrics['token_accuracy'] >= 0.90:
    print('Целевой порог достигнут: token_accuracy >= 0.90')
else:
    print('Целевой порог не достигнут: проверьте формы данных и параметры обучения')

## Что ожидать на практике
- `token_accuracy` обычно растет быстрее `sequence_accuracy`.
- Если `token_accuracy` высокая, а `sequence_accuracy` низкая, модель ошибается в отдельных шагах.
- Резкий разрыв между train и validation может указывать на переобучение.
- Для корректной реализации ожидается достижение `token_accuracy >= 0.90`.


In [ ]:
# Визуализация динамики обучения
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Динамика функции потерь')
plt.xlabel('Эпоха')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.title('Динамика token_accuracy')
plt.xlabel('Эпоха')
plt.legend()

plt.tight_layout()
plt.show()

## Вопросы для самопроверки
1. Почему для `many-to-many` требуется `return_sequences=True`?
2. Чем `sequence_accuracy` отличается от `token_accuracy`?
3. Как форма целевого тензора влияет на корректность обучения?
4. В каких случаях стоит увеличить число эпох, а в каких — уменьшить модель?


## Типичные ошибки (симптом -> причина -> исправление)
- Выход имеет форму `(batch, units)` -> отсутствует `return_sequences=True` -> включить параметр.
- Ошибка сопоставления форм -> `y` не имеет оси признака -> привести к `(N,T,1)`.
- Высокая `token_accuracy`, но низкая `sequence_accuracy` -> локальные ошибки по времени -> проверить разметку и увеличить качество модели.
- Нестабильная валидация -> слишком агрессивные параметры обучения -> уменьшить шаг обучения или упростить модель.
